<a href="https://colab.research.google.com/github/Aws-Abdullah/NLP-Project/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install pyarrow -q

In [33]:
import pandas as pd
import numpy as np
import json
import re
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [34]:
df_arasum = pd.read_csv('arasum.csv', sep='\t', header=None, names=['id', 'text', 'summarizer'])
df_arasum = df_arasum[['text', 'summarizer']]
print("AraSum:", df_arasum.shape)
df_arasum.head()

AraSum: (49604, 2)


,text,summarizer
0,"حقق حزب ""البديل من أجل ألمانيا"" اليميني الشعبو...",تشير آخر استطلاعات الرأي الألمانية إلى تقدم حز...
1,بدأت اليوم الجمعة( 23 أيلول/ سبتمبر 2016 ) في ...,بدأت محكمة ميونخ النظر في اتهامات متعلقة برجل ...
2,قال مسؤولون إن شخصا أصيب إصابة بالغة بعد تعرضه...,أعلن حاكم كارولاينا الشمالية حالة الطورائ في م...
3,أعلن مسؤول في البنتاغون أن جنودا أمريكيين في ش...,قالت وزارة الدفاع الأمريكية (البنتاغون) إن داع...
4,اجتمعت المجموعة الدولية لدعم سوريا على هامش اج...,فشلت الولايات المتحدة وروسيا في الاتفاق على كي...


In [35]:
df_sumtest = pd.read_csv('sumtest.csv')
df_sumtest = df_sumtest[['text', 'summarizer']]
print("sumtest:", df_sumtest.shape)
df_sumtest.head()

sumtest: (8378, 2)


,text,summarizer
0,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...
1,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون...","\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون..."
2,\nاحتضن جناح تونس في القرية الدولية للأفلام بم...,تونس حاضرة من جهة أخرى ستكون تونس حاضرة في قائ...
3,\nشهدت برلين أمس الجمعة افتتاح مسجد فريد من نو...,واستأجرت صاحبة المشروع المحامية والكاتبة سيران...
4,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...


In [36]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            rows.append({
                'text': item.get('text', ''),
                'summarizer': item.get('headline', '')
            })
    return pd.DataFrame(rows)

df_sumar_train = load_jsonl('sumartrain.jsonl')
df_sumar_valid = load_jsonl('sumarvalid.jsonl')
df_sumar_test  = load_jsonl('sumartest.jsonl')

df_sumarabic = pd.concat([df_sumar_train, df_sumar_valid, df_sumar_test], ignore_index=True)
print("SumArabic:", df_sumarabic.shape)
df_sumarabic.head()

SumArabic: (2628, 2)


,text,summarizer
0,اختتمت مساء أول من أمس نهائيات بطولة الإمارات ...,المصري فؤاد الطاهر بطل للشطرنج الديناميكي
1,مينيابوليس (الولايات المتحدة) 3-2-2008 (ا ف ب)...,"حملة انتخابية محمومة قبل ""الثلاثاء الكبير"""
2,أفاد مصدر في شرطة أبوظبي بأن نحو 700 طفل يتواف...,55% من نزلاء «الأحداث» مواطنون
3,قال استشاري أمراض الكلى في مدينة الشيخ خليفة ا...,مركز متخصص في الكلى بمستشفى خليفة
4,بدأت القيادة العامة لشرطة أبوظبي التنسيق لإنشا...,تأهيل ضحايا حوادث المرور في أبوظبي


In [38]:
df_egy = pd.read_parquet('Egy.parquet')
print(df_egy.columns.tolist())
print(df_egy.shape)
df_egy.head()

['text', 'summarized_text', 'source_topics']
(3689, 3)


,text,summarized_text,source_topics
0,كتير من الشباب دلوقتي بيفكروا ألف مرة قبل ما ي...,غلاء المعيشة بيخلي الشباب يقللوا عدد الأطفال ب...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
1,الظروف الاقتصادية الصعبة خلت كتير من الشباب يأ...,الوضع الاقتصادي المتردي بيأجل قرارات الجواز وا...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
2,موجات التضخم المتتالية أثرت بشكل مباشر على قدر...,التضخم بيخلي الأسر تعيد حساباتها في قرارات الإ...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
3,الأسر المصرية بقت بتميل أكتر لإنها تركز على جو...,الأعباء الاقتصادية بتخلي الأسر تركز على جودة ح...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
4,في ظل التحديات الاقتصادية اللي بتواجه الأسر، ب...,الحكومات ممكن تحتاج تقدم دعم وحوافز للأسر عشان...,تأثير الأعباء الاقتصادية على قرارات الإنجاب


In [39]:
import pandas as pd

# Load parquet
df_egy = pd.read_parquet('Egy.parquet')

# Show columns
print("Original columns:", df_egy.columns.tolist())

# Auto-detect columns
cols = df_egy.columns.tolist()

text_col = None
summary_col = None

for c in cols:
    if c.lower() in ['text', 'article', 'content']:
        text_col = c
    if c.lower() in ['summary', 'headline', 'title']:
        summary_col = c

# Fallback (if names are weird)
if text_col is None:
    text_col = cols[0]
if summary_col is None:
    summary_col = cols[1]

print("Using columns:", text_col, "-> text,", summary_col, "-> summarizer")

# Rename and keep only needed columns
df_egy = df_egy.rename(columns={
    text_col: 'text',
    summary_col: 'summarizer'
})

df_egy = df_egy[['text', 'summarizer']]

# Final check
print("Final shape:", df_egy.shape)
df_egy.head()

Original columns: ['text', 'summarized_text', 'source_topics']
Using columns: text -> text, summarized_text -> summarizer
Final shape: (3689, 2)


,text,summarizer
0,كتير من الشباب دلوقتي بيفكروا ألف مرة قبل ما ي...,غلاء المعيشة بيخلي الشباب يقللوا عدد الأطفال ب...
1,الظروف الاقتصادية الصعبة خلت كتير من الشباب يأ...,الوضع الاقتصادي المتردي بيأجل قرارات الجواز وا...
2,موجات التضخم المتتالية أثرت بشكل مباشر على قدر...,التضخم بيخلي الأسر تعيد حساباتها في قرارات الإ...
3,الأسر المصرية بقت بتميل أكتر لإنها تركز على جو...,الأعباء الاقتصادية بتخلي الأسر تركز على جودة ح...
4,في ظل التحديات الاقتصادية اللي بتواجه الأسر، ب...,الحكومات ممكن تحتاج تقدم دعم وحوافز للأسر عشان...


In [40]:
df = pd.concat([df_arasum, df_sumtest, df_sumarabic, df_egy], ignore_index=True)
print("Before cleaning:", df.shape)
df.head()

Before cleaning: (64299, 2)


,text,summarizer
0,"حقق حزب ""البديل من أجل ألمانيا"" اليميني الشعبو...",تشير آخر استطلاعات الرأي الألمانية إلى تقدم حز...
1,بدأت اليوم الجمعة( 23 أيلول/ سبتمبر 2016 ) في ...,بدأت محكمة ميونخ النظر في اتهامات متعلقة برجل ...
2,قال مسؤولون إن شخصا أصيب إصابة بالغة بعد تعرضه...,أعلن حاكم كارولاينا الشمالية حالة الطورائ في م...
3,أعلن مسؤول في البنتاغون أن جنودا أمريكيين في ش...,قالت وزارة الدفاع الأمريكية (البنتاغون) إن داع...
4,اجتمعت المجموعة الدولية لدعم سوريا على هامش اج...,فشلت الولايات المتحدة وروسيا في الاتفاق على كي...


In [45]:



# Remove nulls
df = df.dropna(subset=['text', 'summarizer'])

# Convert to string
df['text'] = df['text'].astype(str)
df['summarizer'] = df['summarizer'].astype(str)

# Remove noise: English letters, @, numbers, punctuation, symbols
def clean_arabic_text(text):
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text'] = df['text'].apply(clean_arabic_text)
df['summarizer'] = df['summarizer'].apply(clean_arabic_text)

# Arabic normalization
def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "ء", text)
    text = re.sub("ئ", "ء", text)
    text = re.sub("ة", "ه", text)
    return text

df['text'] = df['text'].apply(normalize_arabic)
df['summarizer'] = df['summarizer'].apply(normalize_arabic)

# Remove empty rows after cleaning
df = df[(df['text'] != '') & (df['summarizer'] != '')]

# Length filtering
df['text_len'] = df['text'].apply(lambda x: len(x.split()))
df['summary_len'] = df['summarizer'].apply(lambda x: len(x.split()))

df = df[(df['text_len'] >= 10) & (df['summary_len'] >= 3)]

# Remove duplicates
df = df.drop_duplicates(subset=['text', 'summarizer']).reset_index(drop=True)

df['summarizer'] = 'sos ' + df['summarizer'] + ' eos'

print("After preprocessing:", df.shape)
df.head()

After preprocessing: (63924, 4)


,text,summarizer,text_len,summary_len
0,حقق حزب البديل من اجل المانيا اليميني الشعبوي ...,sos تشير اخر استطلاعات الراي الالمانيه الي تقد...,222,34
1,بدات اليوم الجمعه ايلول سبتمبر في ميونيخ محاكم...,sos بدات محكمه ميونخ النظر في اتهامات متعلقه ب...,315,37
2,قال مسءولون ان شخصا اصيب اصابه بالغه بعد تعرضه...,sos اعلن حاكم كارولاينا الشماليه حاله الطوراء ...,378,33
3,اعلن مسءول في البنتاغون ان جنودا امريكيين في ش...,sos قالت وزاره الدفاع الامريكيه البنتاغون ان د...,203,35
4,اجتمعت المجموعه الدوليه لدعم سوريا علي هامش اج...,sos فشلت الولايات المتحده وروسيا في الاتفاق عل...,196,28


In [42]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (51139, 4)
Validation: (6392, 4)
Test: (6393, 4)


In [43]:
tokenizer = Tokenizer(num_words=30000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

train_text_seq = tokenizer.texts_to_sequences(train_df['text'])
val_text_seq   = tokenizer.texts_to_sequences(val_df['text'])
test_text_seq  = tokenizer.texts_to_sequences(test_df['text'])

train_sum_seq = tokenizer.texts_to_sequences(train_df['summarizer'])
val_sum_seq   = tokenizer.texts_to_sequences(val_df['summarizer'])
test_sum_seq  = tokenizer.texts_to_sequences(test_df['summarizer'])

In [44]:
MAX_ARTICLE_LEN = 200
MAX_SUMMARY_LEN = 50

X_train = pad_sequences(train_text_seq, maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post')
X_val   = pad_sequences(val_text_seq,   maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post')
X_test  = pad_sequences(test_text_seq,  maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post')

y_train = pad_sequences(train_sum_seq, maxlen=MAX_SUMMARY_LEN, padding='post', truncating='post')
y_val   = pad_sequences(val_sum_seq,   maxlen=MAX_SUMMARY_LEN, padding='post', truncating='post')
y_test  = pad_sequences(test_sum_seq,  maxlen=MAX_SUMMARY_LEN, padding='post', truncating='post')

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (51139, 200)
y_train: (51139, 50)
X_val: (6392, 200)
y_val: (6392, 50)
X_test: (6393, 200)
y_test: (6393, 50)
